# QuantAccelerator · Research EDA (phase A) · FINRA daily short-sale volume

**Understand → Trust → Research.** This notebook goes one step further than the workbench: once the timing is signed off (G1), the **Research EDA Agent** builds features on the point-in-time research panel, characterizes them against liquidity, size and sector, and proposes a small candidate set that you freeze at **gate G2**, before anything may look at returns.

Phase A is **return-blind** by construction: the panel has no return, price or P&L column, and features are built in a closed language (ratio, change, surprise, peer-relative, residualize, rank), so every feature is reproducible from its definition.

Restart the kernel before running the first cell.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                                    # paths are relative to the project root
os.environ["QA_DATASET"] = "finra"                     # one dataset per kernel, set before the import
os.environ.setdefault("LLM_BASE_URL", "http://127.0.0.1:8000/v1")      # any OpenAI-compatible endpoint
os.environ.setdefault("LLM_MODEL", "Qwen/Qwen2.5-72B-Instruct-AWQ")
from quantaccelerator.notebook import Session
s = Session()

## 1 · Understand (pass 1 only, for speed)
The field meanings are what the PIT agent and the EDA agent need here. Use `s.understand()` for the full profiling pass and the Data Brief.

In [ ]:
card = s.understand(research=False)

## 2 · Trust: gate G1
Read the rule and its plain-words line, then run **one** of the two lines in the next cell.

In [ ]:
s.time()

In [ ]:
# s.approve("why the rule is right")
# s.correct({"source_field": "shvol.Date", "tradable_at": "next_trading_day_open"}, "why")

## 3 · Research EDA
The agent first characterizes every raw field against liquidity (`adv_proxy`), size (`size_float`, `size_assets` from SEC filings, point-in-time by filing date) and sector, then builds representations where a dependence calls for one, compares them, and proposes candidates. A dependence is a question, not a verdict: each finding lists the explanations it cannot yet tell apart (measurement bias, economic scale, genuine interaction).

In [ ]:
fs = s.explore()

In [ ]:
s.features()   # the Feature Report: candidates, definitions, remaining dependences, findings with figures

## 4 · Gate G2: freeze the candidate set
Freeze all candidates, or a subset with `keep=[...]`. The frozen set gets a hash (its definitions plus the panel) and its values are written to `features/<hash>.parquet` in the session folder. Predictive work (phase B, with returns) may only use a frozen set.

In [ ]:
# s.freeze_features(note="why")                 # all candidates
# s.freeze_features(keep=["..."], note="why")  # a subset
# s.reject_features("why")

In [ ]:
s.export_html()